### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

d:\2026-courses\agenticai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000249A381D3F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000249A381DAE0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [4]:
model_with_structure= model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000249A381D3F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000249A381DAE0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out o

In [6]:
model.invoke("Provide me the details of the movie 'The Dark Knight'")

AIMessage(content='<think>\nOkay, so I need to find out the details about the movie \'The Dark Knight\'. Let me start by recalling what I know. I think it\'s part of the Batman series, directed by Christopher Nolan. The main characters are Batman, the Joker, and maybe some others like Gotham City\'s mayor or Harvey Dent. The actor who plays Batman is Christian Bale, right? And the Joker was played by a famous actor... wasn\'t it Heath Ledger? He did a really good job and even won a posthumous Oscar for it. \n\nThe movie came out in 2008, I believe, and it\'s part of the Batman reboot trilogy. The director, Christopher Nolan, is known for making complex, intelligent films with great action. The movie\'s plot probably involves Batman trying to stop the Joker, who is a chaotic villain. There\'s a character named Harvey Dent who is the district attorney, and maybe he turns into Two-Face later? I think there\'s a tragic twist in the movie where something happens to Dent that changes him. \n

In [5]:
model_with_structure.invoke("Provide me the details of the movie 'The Dark Knight'")

Movie(title='The Dark Knight', year=2008, director='Christopher Nolan', rating=9.0)

### MEssage output alongside parsed structure

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide me the details of the movie 'The Dark Knight'")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the details of the movie 'The Dark Knight'. Let me see what information I need to provide. The available function is called Movie, which requires the title, year, director, and rating. \n\nFirst, I need to confirm the title. 'The Dark Knight' is the correct title. The year it was released was 2008. The director is Christopher Nolan. As for the rating, I think it's around 9.0 on IMDb, but maybe I should check a reliable source to be sure. However, since I don't have real-time data access, I'll go with the commonly known rating.\n\nWait, the function parameters specify a rating out of 10. The user might expect a decimal value. Let me make sure all required fields are included: title, year, director, and rating. I have all of those. So I'll structure the tool call with those parameters. Let me double-check the spelling of the director's name and the movie title. Everything looks correct. Al

### Nested Structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide me the details of the movie 'The Dark Knight'")
response

MovieDetails(title='The Dark Knight', year=2008, cast=[Actor(name='Christian Bale', role='Bruce Wayne / Batman'), Actor(name='Heath Ledger', role='Joker'), Actor(name='Aaron Eckhart', role='Harvey Dent'), Actor(name='Michael Caine', role='Alfred')], genres=['Action', 'Crime', 'Drama'], budget=150.0)